# Heart Sound Classification using Machine Learning

## Objective
Build a machine learning model that classifies heart sounds into different medical conditions using audio recordings.

## Dataset
HLS-CMDS: Heart and Lung Sounds Dataset (UCI Repository)

We use:
- HS.csv (labels + metadata)
- WAV files (audio signals)

Each Heart Sound ID corresponds to a .wav file.

In [ ]:
import pandas as pd
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Load the dataset
df = pd.read_csv("/dataset/HS.csv")

print(df.head())
print(df["Heart Sound Type"].value_counts())

In [ ]:
# Clean data
counts = df["Heart Sound Type"].value_counts()

valid_classes = counts[counts >= 5].index
df = df[df["Heart Sound Type"].isin(valid_classes)]

print("Final classes:\n", df["Heart Sound Type"].value_counts())

In [ ]:
# Map audio filepaths
df["Heart Sound ID"] = df["Heart Sound ID"].astype(str).str.strip()

base_path = "/dataset/HS"

df["filepath"] = df["Heart Sound ID"].apply(
    lambda x: os.path.join(base_path, x + ".wav")
)

In [ ]:
# feature extraction
X = []
y = []

valid_count = 0

for _, row in df.iterrows():

    path = row["filepath"]
    label = row["Heart Sound Type"]

    if not os.path.exists(path):
        print("Missing:", path)
        continue

    try:
        audio, sr = librosa.load(path, sr=22050)

        mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20), axis=1)

        zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio))
        rms = np.mean(librosa.feature.rms(y=audio))
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
        bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))

        features = np.hstack([mfcc, zcr, rms, centroid, bandwidth])

        X.append(features)
        y.append(label)

        valid_count += 1

    except Exception as e:
        print("Error:", path, e)

print("Valid samples loaded:", valid_count)

In [ ]:
# Converting to ml format
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Model pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

In [ ]:
# Train Model
model.fit(X_train, y_train)

In [ ]:
# Test Performance Evaluation
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=model.classes_
)

disp.plot(xticks_rotation=45)
plt.show()

In [ ]:
# Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

In [ ]:
# Train on full data
model.fit(X, y)

In [ ]:
# single prediction function
def predict_heart(file_path):

    audio, sr = librosa.load(file_path, sr=22050)

    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20), axis=1)

    zcr = np.mean(librosa.feature.zero_crossing_rate(y=audio))
    rms = np.mean(librosa.feature.rms(y=audio))
    centroid = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))

    features = np.hstack([mfcc, zcr, rms, centroid, bandwidth]).reshape(1, -1)

    return model.predict(features)[0]

In [ ]:
print(predict_heart("dataset/HS/F_N_RC.wav"))